In [12]:
# imports
import os
from langchain_community.vectorstores import FAISS
from langchain_cohere import ChatCohere, CohereEmbeddings
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_core.documents import Document

In [13]:
# load the api keys.
os.environ['COHERE_API_KEY'] = "cohere_5iQczItcuqnmN02zj7Tue4cEvswgbT6d1x8uQTwQ3M1gJx"

In [14]:
# Recreate the document objects from the previous data
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [21]:
# initialize the embedding models.
embedding_model = CohereEmbeddings(
    model = "embed-english-v3.0"
)

# create the vector store. 
vector_store = FAISS.from_documents(
    documents = docs, 
    embedding = embedding_model
)

Retrying langchain_cohere.embeddings.CohereEmbeddings.embed_with_retry.<locals>._embed_with_retry in 4.0 seconds as it raised ConnectError: [Errno 11001] getaddrinfo failed.
Retrying langchain_cohere.embeddings.CohereEmbeddings.embed_with_retry.<locals>._embed_with_retry in 4.0 seconds as it raised ConnectError: [Errno 11001] getaddrinfo failed.


ConnectError: [Errno 11001] getaddrinfo failed

In [16]:
# create the base retriever.
base_retriever = vector_store.as_retriever(search_kwargs = {'k' : 2})

In [17]:
# Set up the compressor using an LLM
# initialize the chat model. 
chat_model = ChatCohere(
    model="command-a-03-2025",
    temperature=0
)
compressor = LLMChainExtractor.from_llm(chat_model)

In [19]:
# create the contextual retrivers.
compression_retriever = ContextualCompressionRetriever(
    base_retriever = base_retriever, 
    base_compressor = compressor
)

In [20]:
# Query the retriever
query = "What is photosynthesis?"
compressed_results = compression_retriever.invoke(query)

Retrying langchain_cohere.embeddings.CohereEmbeddings.embed_with_retry.<locals>._embed_with_retry in 4.0 seconds as it raised ConnectError: [Errno 11001] getaddrinfo failed.
Retrying langchain_cohere.embeddings.CohereEmbeddings.embed_with_retry.<locals>._embed_with_retry in 4.0 seconds as it raised ConnectError: [Errno 11001] getaddrinfo failed.


ConnectError: [Errno 11001] getaddrinfo failed

In [ ]:
for i, doc in enumerate(compressed_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)
